In [43]:
# titanic-without-using-pipeline

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

In [44]:
df = pd.read_csv("train.csv")
df.sample(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
225,226,0,3,"Berglund, Mr. Karl Ivar Sven",male,22.0,0,0,PP 4348,9.3500,NaN,S
402,403,0,3,"Jussila, Miss. Mari Aina",female,21.0,1,0,4137,9.8250,NaN,S
221,222,0,2,"Bracken, Mr. James H",male,27.0,0,0,220367,13.0000,NaN,S
234,235,0,2,"Leyson, Mr. Robert William Norman",male,24.0,0,0,C.A. 29566,10.5000,NaN,S
380,381,1,1,"Bidois, Miss. Rosalie",female,42.0,0,0,PC 17757,227.5250,NaN,C
662,663,0,1,"Colley, Mr. Edward Pomeroy",male,47.0,0,0,5727,25.5875,E58,S
251,252,0,3,"Strom, Mrs. Wilhelm (Elna Matilda Persson)",female,29.0,1,1,347054,10.4625,G6,S
185,186,0,1,"Rood, Mr. Hugh Roscoe",male,NaN,0,0,113767,50.0000,A32,S
670,671,1,2,"Brown, Mrs. Thomas William Solomon (Elizabeth ...",female,40.0,1,1,29750,39.0000,NaN,S
581,582,1,1,"Thayer, Mrs. John Borland (Marian Longstreth M...",female,39.0,1,1,17421,110.8833,C68,C


In [45]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)
df.sample(10)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
535,1,2,female,7.0,0,2,26.25,S
416,1,2,female,34.0,1,1,32.50,S
21,1,2,male,34.0,0,0,13.00,S
607,1,1,male,27.0,0,0,30.50,S
889,1,1,male,26.0,0,0,30.00,C
604,1,1,male,35.0,0,0,26.55,C
880,1,2,female,25.0,0,1,26.00,S
430,1,1,male,28.0,0,0,26.55,S
339,0,1,male,45.0,0,0,35.50,S
282,0,3,male,16.0,0,0,9.50,S


In [46]:
x_train,x_test,y_train,y_test=train_test_split(df.drop(columns=['Survived']),df['Survived'],test_size=0.2,random_state=42)

x_train.shape

(712, 7)

In [47]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [48]:
# imputation

si_age=SimpleImputer()
si_embarked=SimpleImputer(strategy='most_frequent')

x_train_age=si_age.fit_transform(x_train[['Age']])
x_train_embarked=si_embarked.fit_transform(x_train[['Embarked']])
x_test_age=si_age.transform(x_test[['Age']])
x_test_embarked=si_embarked.transform(x_test[['Embarked']])

x_train_embarked.shape

(712, 1)

In [49]:
# Onehotencoding on sex and embarked

ohe=OneHotEncoder(sparse_output=False,handle_unknown='ignore')
ohe_embarked=OneHotEncoder(sparse_output=False,handle_unknown='ignore')

x_train_sex=ohe.fit_transform(x_train[['Sex']])
x_train_emb=ohe_embarked.fit_transform(x_train_embarked)

x_test_sex=ohe.transform(x_test[['Sex']])
x_test_emb=ohe_embarked.transform(x_test_embarked)

x_train_age.shape,x_train_embarked

((712, 1),
 array([['S'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['C'],
        ['S'],
        ['C'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['C'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['C'],
        ['C'],
        ['S'],
        ['S'],
        ['C'],
        ['S'],
        ['C'],
        ['S'],
        ['Q'],
        ['Q'],
        ['S'],
        ['S'],
        ['S'],
        ['C'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['C'],
        ['Q'],
        ['C'],
        ['S'],
        ['S'],
        ['S'],
        ['S'],
        ['C'],
        ['Q'],
        ['S'],
        ['S'],
        ['C'],
        ['S'],
        ['C'],

In [51]:
x_train_rem=x_train.drop(columns=['Sex','Embarked','Age'])
x_test_rem=x_test.drop(columns=['Sex','Embarked','Age'])

x_train_trans=np.concatenate((x_train_rem,x_train_age,x_train_sex,x_train_emb),axis=1)
x_test_trans=np.concatenate((x_test_rem,x_test_age,x_test_sex,x_test_emb),axis=1)

x_train_trans.shape

(712, 10)

In [52]:
print(x_train_age.dtype)
print(x_train_sex.dtype)
print(x_train_embarked.dtype)

float64
float64
object


In [53]:
dtc=DecisionTreeClassifier()
dtc.fit(x_train_trans,y_train)
pred=dtc.predict(x_test_trans)

In [54]:
from sklearn.metrics import accuracy_score

print("accuracy score is:",accuracy_score(y_test,pred))

accuracy score is: 0.776536312849162


In [56]:
import pickle

pickle.dump(ohe,open('models/ohe_sex.pkl','wb'))
pickle.dump(ohe_embarked,open('models/ohe_embarked.pkl','wb'))
pickle.dump(dtc,open('models/ohe_clf.pkl','wb'))